In [ ]:
# Load Required Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import os

DATA_PATH = os.getenv("DATA_PATH", "D:/Dimple-AI/Guvi_Projects/Guvi_Project6_MutiAgent/")
MODEL_PATH = os.getenv("MODEL_PATH", "D:/Dimple-AI/Guvi_Projects/Guvi_Project6_MutiAgent/models")

In [ ]:
raw_data = pd.read_csv(DATA_PATH + "Data/support_tickets_10k.csv")
print("Sentiments : ", raw_data["sentiment"].unique())
print(raw_data.value_counts("sentiment"))

Sentiments :  <ArrowStringArray>
['Negative', 'Positive', 'Very Negative', 'Neutral', 'Slightly Negative']
Length: 5, dtype: str
sentiment
Negative             3077
Neutral              2732
Slightly Negative    1800
Very Negative        1543
Positive              848
Name: count, dtype: int64


In [4]:
raw_data.duplicated().sum()
raw_data['ticket_category'].value_counts()

ticket_category
Refund & Return             2051
Delivery Issue              1950
Product Issue               1948
Payment Issue               1474
Account & Login             1030
Seller & Product Listing     830
App & Website Issue          717
Name: count, dtype: int64

In [6]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dimpl\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dimpl\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\dimpl\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\dimpl\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [10]:
def clean_data(text):
    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Tokenize the text
    tokens = nltk.word_tokenize(text)
    # Remove stop words
    tokens = [word for word in tokens if word not in stop_words]
    # Lemmatization
    lemmatizer = nltk.WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

In [37]:
#Clean Data
raw_data['cleaned_ticket_text'] = raw_data['ticket_text'].apply(clean_data)
cleanData = raw_data[['cleaned_ticket_text', 'ticket_category', 'sentiment']]
display(cleanData.head(5))

,cleaned_ticket_text,ticket_category,sentiment
0,lipstick received order ord stopped working da...,Product Issue,Negative
1,shoe rack received order ord scratch dent look...,Product Issue,Negative
2,want exchange conditioner order ord different ...,Refund & Return,Positive
3,makeup brush set order ord appears counterfeit...,Product Issue,Very Negative
4,product listing lipstick completely misleading...,Product Issue,Neutral


In [ ]:
#generate word cloud for each sentiment

import matplotlib.pyplot as plt

for sentiment in cleanData['sentiment'].unique():
    text = ' '.join(cleanData[cleanData['sentiment'] == sentiment]['cleaned_ticket_text'])
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(f'Word Cloud for {sentiment}')
    plt.axis('off')
    plt.show()

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [48]:
def get_tuned_models():
    """
    Returns a dictionary of machine learning models with predefined hyperparameters.
    """
    tuned_models = {
        "Logistic Regression": LogisticRegression(random_state=42, solver='lbfgs', max_iter=200),
        "LinearSVC": LinearSVC(random_state=42, C=0.5, max_iter=1000),
        "MultinomialNB": MultinomialNB(alpha=0.1), # Alpha is a smoothing parameter
        "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10, min_samples_leaf=5),
       }
    return tuned_models

In [39]:
#generate code to get records from each sentiment. number to record per sentiment is value count of minimun sentiment category ,keep sentiment column in the dataframe and reset index of the dataframe after sampling


min_count = cleanData['sentiment'].value_counts().min()
cleanData = cleanData.groupby('sentiment').sample(
                      n=min_count,
                      random_state=42
                    ).reset_index(drop=True)

In [41]:
#Split the data into training and testing sets
# Split data for sentiment Classification
x_train, x_test, y_train, y_test = train_test_split(cleanData['cleaned_ticket_text'],
                                                    cleanData['sentiment'],
                                                    stratify=cleanData['sentiment'],
                                                    test_size=0.2, random_state=42)

tfidf_vectorizer_sentiment = TfidfVectorizer(max_features=2000,ngram_range=(1,2))
x_train_tfidf = tfidf_vectorizer_sentiment.fit_transform(x_train)


In [ ]:
# save sentiment vectorizer in pickle file
import pickle
with open(DATA_PATH + 'models/tfidf_vectorizer_sentiment.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer_sentiment, f)



In [46]:
#TF- IDF Vectorization and Model Training

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
#from sklearn import BertClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

BEST_MODEL = None

In [ ]:
def train_evaluate_sentiment_model(tuned_models, x_train, x_test, y_train, y_test):
    """
    Trains each model in the dictionary and returns evaluation results.
    """
    results = []
    trained_models = {}

    for model_name, model in tuned_models.items():
        # Training
        model.fit(x_train, y_train)

        # Predictions
        y_train_pred = model.predict(x_train)
        y_test_pred = model.predict(x_test)

        # Metrics
        train_accuracy = accuracy_score(y_train, y_train_pred)
        test_accuracy = accuracy_score(y_test, y_test_pred)
        # Fix: Set output_dict=True to return a dictionary instead of a string
        report = classification_report(y_test, y_test_pred, output_dict=True)

        # Store trained models
        trained_models[model_name] = model

        results.append({
            'Model': model_name,
            'Train Accuracy': train_accuracy,
            'Test Accuracy' : test_accuracy,
            'precision': report.get('weighted avg', {}).get('precision', 0),
            'recall': report.get('weighted avg', {}).get('recall', 0),
            'f1-score': report.get('weighted avg', {}).get('f1-score', 0)
        })

        # Code tocheck if model predicting all cxatagory or not
        print("\nModel:", model_name)

        print("Actual classes:")
        print(sorted(y_test.unique()))

        print("Predicted classes:")
        print(sorted(set(y_train_pred)))

        print("Missing predicted classes:")
        print(set(y_test.unique()) - set(y_train_pred))
      # end of the code
    return results,trained_models

# 1. Vectorize the test set using the fitted vectorizer
x_test_tfidf = tfidf_vectorizer_sentiment.transform(x_test)


# 2. Get the tuned models
tuned_models = get_tuned_models()

# 3. Train and Evaluate - Sentiment Classification
evaluation_data_Sentiment,models = train_evaluate_sentiment_model(tuned_models, x_train_tfidf, x_test_tfidf, y_train, y_test)


# 5. Display Results in a DataFrame
sentiment_results_df = pd.DataFrame(evaluation_data_Sentiment)
sentiment_results_df.set_index('Model', inplace=True)

#display("Category Classification Model\n"+ category_results_df)
print("Sentiment Classification Model")
display(sentiment_results_df.sort_values(by='f1-score', ascending=False))

#save bestModel in  drive
bestModelName = sentiment_results_df.sort_values(
                                by='f1-score',  ascending=False).index[0]


bestModel   = models[bestModelName]

BEST_MODEL = "Sentiment_BestModel_" + bestModelName

#save best model name in .env file
with open(".env", "a") as f:
    f.write(f"\nBEST_MODEL={BEST_MODEL}\n")

with open(MODEL_PATH + "/"+BEST_MODEL+".pkl", "wb") as f:
    pickle.dump(bestModel, f)


Model: Logistic Regression
Actual classes:
['Negative', 'Neutral', 'Positive', 'Slightly Negative', 'Very Negative']
Predicted classes:
['Negative', 'Neutral', 'Positive', 'Slightly Negative', 'Very Negative']
Missing predicted classes:
set()

Model: LinearSVC
Actual classes:
['Negative', 'Neutral', 'Positive', 'Slightly Negative', 'Very Negative']
Predicted classes:
['Negative', 'Neutral', 'Positive', 'Slightly Negative', 'Very Negative']
Missing predicted classes:
set()

Model: MultinomialNB
Actual classes:
['Negative', 'Neutral', 'Positive', 'Slightly Negative', 'Very Negative']
Predicted classes:
[np.str_('Negative'), np.str_('Neutral'), np.str_('Positive'), np.str_('Slightly Negative'), np.str_('Very Negative')]
Missing predicted classes:
set()

Model: Random Forest
Actual classes:
['Negative', 'Neutral', 'Positive', 'Slightly Negative', 'Very Negative']
Predicted classes:
['Negative', 'Neutral', 'Positive', 'Slightly Negative', 'Very Negative']
Missing predicted classes:
set()
S

,Train Accuracy,Test Accuracy,precision,recall,f1-score
Model,,,,,
MultinomialNB,0.620283,0.537736,0.525879,0.537736,0.522306
Logistic Regression,0.619104,0.528302,0.517374,0.528302,0.514820
LinearSVC,0.640330,0.527123,0.514956,0.527123,0.513017
Random Forest,0.541568,0.510613,0.492842,0.510613,0.471572
